In [1]:
!git clone https://github.com/jjeong3150/AIFFEL_final_pjt_book-recommendation-agent

fatal: destination path 'AIFFEL_final_pjt_book-recommendation-agent' already exists and is not an empty directory.


# Main — CSR × User Simulation 오케스트레이션

```
builder_mvp.ipynb  →  app, initial_state, config 제공 (while 루프 셀 제거 필요)
user_sim.ipynb     →  UserSimAgent, PERSONA_TEMPLATES
main.ipynb         →  Queue 기반으로 두 시스템 연결
```

**주의**: `builder_mvp.ipynb`의 마지막 셀 (while 루프)은 주석 처리하거나 삭제하세요.  
루프 로직은 이 노트북이 담당합니다.

## 1. 노트북 로드

In [2]:
# CSR 시스템 로드 (app, initial_state, config 가 네임스페이스로 올라옴)
%run /content/AIFFEL_final_project_peekabook/research/src/pipeline/builder_mvp.ipynb

/tmp/ipykernel_1855/2258370923.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
/tmp/ipykernel_1855/2258370923.py:64: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 12.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

/tmp/ipykernel_1855/896750007.py:41: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


In [4]:
# User Sim 로드 (UserSimAgent, PERSONA_TEMPLATES 가 네임스페이스로 올라옴)
%run /content/AIFFEL_final_project_peekabook/research/src/simulation/user_sim_test.ipynb

사용 가능한 페르소나: ['직장인_SF팬', '대학생_문학팬', '중년_역사_비문학']
UserSimAgent 정의 완료


## 2. 오케스트레이션 설정

In [5]:
import queue
import threading
import json
import copy
from langgraph.types import Command

# 통신 큐
user_to_csr = queue.Queue()   # UserSim → CSR (사용자 응답)
csr_to_user = queue.Queue()   # CSR → UserSim (질문 or 완료 신호)

# 결과 수집
eval_results = []

print("큐 초기화 완료")

큐 초기화 완료


## 3. CSR 실행 함수

`input()` 대신 `user_to_csr` 큐에서 읽어옵니다.

In [6]:
def run_csr(thread_id: str):
    """
    CSR LangGraph 앱을 실행하며 interrupt 발생 시
    input() 대신 큐에서 사용자 응답을 가져옵니다.
    """
    session_config = {"configurable": {"thread_id": thread_id}}
    state = copy.deepcopy(initial_state)
    result = app.invoke(state, config=session_config)

    while True:
        if "__interrupt__" in result:
            question = result["__interrupt__"][0].value
            csr_to_user.put(question)           # UserSim에 질문 전달
            user_input = user_to_csr.get()      # UserSim 응답 대기 (블로킹)
            result = app.invoke(Command(resume=user_input), config=session_config)
        else:
            csr_to_user.put({"__done__": True, "result": result})
            break

print("run_csr 정의 완료")

run_csr 정의 완료


## 4. UserSim 실행 함수

In [7]:
def run_user_sim(persona: dict, result_collector: list):
    """
    UserSimAgent를 실행하며 CSR 질문에 자동 응답합니다.
    세션이 끝나면 결과를 result_collector에 추가합니다.
    """
    agent = UserSimAgent(persona=persona, verbose=True)

    while True:
        message = csr_to_user.get()  # CSR 메시지 대기 (블로킹)

        if isinstance(message, dict) and message.get("__done__"):
            csr_result = message["result"]
            result_collector.append({
                "persona":         persona,
                "profile":         csr_result.get("profile", {}),
                "summary":         csr_result.get("summary", ""),
                "recommendations": csr_result.get("recommendations", []),
                "final_message": (
                    csr_result["messages"][-1].content
                    if csr_result.get("messages") else ""
                ),
                "conversation": agent.get_history()
            })
            break

        response = agent.answer(message)
        user_to_csr.put(response)

print("run_user_sim 정의 완료")

run_user_sim 정의 완료


## 5. 단일 세션 실행

In [8]:
# 페르소나 선택
persona   = PERSONA_TEMPLATES["중년_역사_비문학"]
thread_id = "eval_session_001"

t_csr  = threading.Thread(target=run_csr,      args=(thread_id,))
t_user = threading.Thread(target=run_user_sim,  args=(persona, eval_results))

t_csr.start()
t_user.start()

t_csr.join()
t_user.join()

print("\n" + "="*50)
print("세션 종료")
print("="*50)


[Turn 1]
  CSR  : 현재 독서를 통해 어떤 목표를 이루고 싶으신가요?
  USER : 주로 수업에서 활용할 수 있는 흥미로운 역사 이야기나 사례를 찾고 싶어요. 학생들이 역사에 더 흥미를 느끼도록 도와주고 싶거든요.

[Turn 2]
  CSR  : 최근에 읽었던 책 중에서 가장 흥미로웠던 장르는 무엇인가요?
  USER : 최근에 읽었던 책 중에서 역사 비문학이 가장 흥미로웠어요. 특히 과거 인류의 진화나 문화 발전에 대한 이야기가 정말 매력적이었죠.

[Turn 3]
  CSR  : 책을 읽을 때, 주로 어떤 방식으로 내용을 이해하고 흡수하는 것을 선호하시나요? 예를 들어, 천천히 깊이 있게 읽는 것을 좋아하시나요, 아니면 빠르게 읽고 전체적인 흐름을 파악하는 것을 선호하시나요?
  USER : 주로 천천히 깊이 있게 읽는 걸 선호해요. 중요한 내용이나 흥미로운 부분은 메모도 하면서, 학생들에게 설명할 때 활용할 수 있도록 잘 이해하려고 노력하거든요.

[Turn 4]
  CSR  : 어떤 난이도의 책을 선호하시나요? 초급, 중급, 고급 중에서 선택해 주실 수 있나요?
  USER : 중급 정도의 난이도를 선호해요. 너무 어렵지 않으면서도 깊이 있는 통찰을 제공하는 책이 좋더라고요. 수업에서도 쉽게 설명할 수 있어서 유용하죠.

[Turn 5]
  CSR  : 최근에 어떤 감정이나 상황에서 책을 읽고 싶으신가요?
  USER : 최근에는 학생들에게 긍정적인 영향을 줄 수 있는 책을 읽고 싶어요. 수업에 활용할 수 있는 흥미로운 역사 이야기를 찾아서, 학생들의 관심을 끌어내고 싶습니다.

[Turn 6]
  CSR  : 이전에 중급 난이도의 역사 비문학 책을 읽어본 적이 있으신가요? 그 책의 제목과 내용에 대한 감상이나 소감을 공유해 주실 수 있을까요?
  USER : 네, '사피엔스'를 읽어봤어요. 인류의 역사와 진화 과정을 통찰력 있게 다루고 있어서 정말 인상 깊었어요. 특히 인류가 어떻게 사회를 발전시켰는지에 대한 이야기를 통해 학생들에게

## 6. 결과 확인

In [9]:
if eval_results:
    r = eval_results[-1]

    print("[페르소나]")
    print(json.dumps(r["persona"], ensure_ascii=False, indent=2))

    print("\n[추출된 프로필]")
    print(json.dumps(r["profile"], ensure_ascii=False, indent=2))

    print("\n[요약]")
    print(r["summary"])

    print("\n[책 리스트]")
    print(r["recommendations"])

    print("\n[추천 결과]")
    print(r["final_message"])
else:
    print("결과 없음")

[페르소나]
{
  "age_group": "40대 중반",
  "job": "중학교 역사 교사",
  "favorite_genre": "역사, 교양 비문학",
  "reading_frequency": "한 달에 3~4권",
  "mood": "수업에 활용할 수 있는 흥미로운 역사 이야기",
  "disliked_genre": "공포, 오컬트",
  "reading_experience": "유발 하라리의 사피엔스를 인상 깊게 읽음"
}

[추출된 프로필]
{
  "reading_goal": "수업에서 활용할 수 있는 흥미로운 역사 이야기나 사례",
  "preferred_genre": "역사 비문학",
  "reading_style": "천천히 깊이 있게 읽는 것을 선호하며, 중요한 내용이나 흥미로운 부분을 메모하고 학생들에게 설명할 때 활용하려고 노력함",
  "difficulty_level": "중급",
  "current_context": "학생들에게 긍정적인 영향을 줄 수 있는 책, 수업에 활용할 수 있는 흥미로운 역사 이야기"
}

[요약]
사용자는 수업에서 활용할 수 있는 흥미로운 역사 이야기나 사례를 찾고 있으며, 역사 비문학 장르를 선호합니다. 천천히 깊이 있게 읽는 스타일로, 중요한 내용이나 흥미로운 부분을 메모하여 학생들에게 설명하는 데 활용하려고 합니다. 중급 난이도의 책을 선호하며, 학생들에게 긍정적인 영향을 줄 수 있는 내용을 중시하고 있습니다.

[책 리스트]
[{'title': '한 번에 끝내는 중학 세계사 1 고대와 중세', 'author': '김상훈', 'isbn': '9791188762255', 'reason': '이 책은 중학교 역사 교과 과정을 충실히 반영하여 역사적 사건의 인과관계를 쉽게 설명합니다. 사용자가 학생들에게 설명할 때 유용한 내용이 많아, 깊이 있는 독서와 메모를 통해 수업에 활용하기 적합합니다.'}, {'title': '청소년을 위한 한국사 - 년을 위한 역사 교양 3', 'author': '백유선', 'is

## 7. 다중 페르소나 배치 평가

여러 페르소나를 순차적으로 평가합니다.  
(동시 실행 시 큐 혼선이 생기므로 순차 실행)

In [10]:
def run_batch_evaluation(persona_dict: dict, base_thread_id: str = "batch"):
    global user_to_csr, csr_to_user
    batch_results = []

    for i, (name, persona) in enumerate(persona_dict.items()):
        print(f"\n{'='*50}")
        print(f"[{i+1}/{len(persona_dict)}] 페르소나: {name}")
        print(f"{'='*50}")

        # 세션마다 큐 초기화
        user_to_csr = queue.Queue()
        csr_to_user = queue.Queue()

        session_results = []
        thread_id = f"{base_thread_id}_{i}"

        t_csr  = threading.Thread(target=run_csr,      args=(thread_id,))
        t_user = threading.Thread(target=run_user_sim,  args=(persona, session_results))

        t_csr.start()
        t_user.start()
        t_csr.join()
        t_user.join()

        if session_results:
            result = session_results[0]
            result["persona_name"] = name
            batch_results.append(result)
            print(f"  완료: 추천 {len(result.get('recommendations', []))}건")

    print(f"\n배치 평가 완료: {len(batch_results)}/{len(persona_dict)} 성공")
    return batch_results


# 실행 (주석 해제 시 전체 페르소나 평가)
# all_results = run_batch_evaluation(PERSONA_TEMPLATES)